# Medallion Architecture

#### Initialize

In [0]:

%sql
use catalog dbacademy;
use schema get_started_de;
select current_catalog(), current_schema();

In [0]:
%sql
LIST '/Volumes/dbacademy/get_started_de/myfiles'

#### Bronze Table

In [0]:
%sql
CREATE table if not exists employees_bronze(
    ID INT,
    FirstName STRING,
    Country STRING,
    Role STRING
);

In [0]:
%sql
Select * from employees_bronze;

In [0]:
result = spark.sql("""
                   COPY into employees_bronze
                   from '/Volumes/dbacademy/get_started_de/myfiles'
                   fileformat = CSV
                   format_options ('header' = 'true', 'inferSchema' = 'true')""")
result.display()

#### Silver Table

In [0]:
%sql
CREATE or REPLACE TABLE employees_silver AS
SELECT 
    ID, 
    FirstName, 
    Country,
    upper(Role) AS Role,
    current_timestamp() as processed_timestamp,
    current_date() as processed_date
FROM employees_bronze;


In [0]:
%sql
select * from employees_silver;

#### Gold Table

Creating view

In [0]:
%sql
Create or replace temp view temp_total_rows as
select 
 Role,
 count(*) as TotalEmployees
from employees_silver
group by Role;

In [0]:
%sql
select * from temp_total_rows;

In [0]:
%sql
Create table if not exists employees_gold (
    Role STRING,
    TotalEmployees INT
);

In [0]:
%sql
insert overwrite employees_gold
select * from temp_total_rows;


In [0]:
%sql
Select * from employees_gold;